In [ ]:
import logging
from pathlib import Path
from datetime import datetime

from pyhrrrzarr.schema import csv_to_locations
from pyhrrrzarr.hrrr.schema import HRRRVariable, VariableName, Level 
from pyhrrrzarr.hrrrzarr.requests import create_requests
from pyhrrrzarr.hrrrzarr.fetch import get_all_request_values
from pyhrrrzarr.hrrrzarr.postprocess import requests_to_df, add_wind_speed_and_direction


logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

In [ ]:
from pyhrrrzarr.hrrrzarr.schema import ZarrId, Request


# download every chunk for specific variable
variable = HRRRVariable(name=VariableName.TMP, level=Level._SURFACE)
hour = datetime(2024,12,1,12,00,00)
zarr_id = ZarrId(
    run_hour=hour,
    type_level=variable.type_level,
    variable=variable,
    type_model=variable.type_model, 
)

In [ ]:
import pydantic
import numpy as np

from pyhrrrzarr.hrrrzarr.schema import RequestWholeChunk
    

def requests_for_all_chunks(zarr_id: ZarrId) -> list[RequestWholeChunk]:
    chunk_id_strs = [f"{y}.{x}" for y in range(0, 7) for x in range(0, 12)]
    return [RequestWholeChunk(zarr_id=zarr_id, chunk_id_str=chunk_id_str) for chunk_id_str in chunk_id_strs]





In [ ]:
requests = requests_for_all_chunks(zarr_id)
updated_requests = await get_all_request_values(requests, batch_size=10)

In [ ]:
updated_requests[0].value.shape

In [ ]:
import xarray as xr

from pyhrrrzarr.hrrrzarr.chunk import get_chunk_index


def create_chunk_dataarrays(requests: list[RequestWholeChunk], chunk_index: xr.DataArray) -> list[xr.DataArray]:
    dataarrays = []
    for request in requests:
        data = request.value
        chunk_index_da = chunk_index.where(chunk_index.chunk_id==request.chunk_id(), drop=True)
        # hrrrzarr col index 0-11, final col has with 149, not 150.
        if chunk_index_da.x.shape[0] < data.shape[1]:
            print(request.chunk_id(), data.shape)
            data = data[:, :chunk_index_da.x.shape[0]]
        
        try:    
            chunk_index_da[request.zarr_id.variable.name.value] = xr.DataArray(
                data=data,
                dims=['y', 'x'],
                coords={'y': chunk_index_da.y, 'x': chunk_index_da.x},
            )
            chunk_index_da.coords['time'] = request.zarr_id.run_hour
            dataarrays.append(chunk_index_da)
        except ValueError as e:
            print(e)
    return dataarrays


chunk_das = create_chunk_dataarrays(updated_requests, chunk_index=get_chunk_index())

In [ ]:
import holoviews as hv
import hvplot.xarray


hv.extension('bokeh')
hv.plotting.bokeh.element.ElementPlot.active_tools = ['box_zoom']

conus_data = xr.combine_by_coords(chunk_das,)
conus_data.TMP.hvplot(cmap='viridis', width=800, height=600, x="x", y="y", title=f'Temperature (K), {np.datetime64(conus_data.time.item(), "ns")}')

In [ ]:
import rioxarray


def datetime_from_np64(time: np.datetime64 | int) -> datetime:
    if isinstance(time, int):
        time = np.datetime64(time, "ns")
    ts = (time - np.datetime64('1970-01-01T00:00:00Z')) / np.timedelta64(1, 's')
    return datetime.utcfromtimestamp(ts)


def create_geotiff_output_path(ds: xr.Dataset, var: str, output_root_path: Path) -> Path:
    """
    create output path with dir by var, year, month, day. Make the dir if not exists
    """
    time = datetime_from_np64(ds.time.item())
    output_filename = f"{var}_{time.year}_{time.month:02d}_{time.day:02d}_{time.hour:02d}.tif"
    output_path = output_root_path / f"{var}" / f"{time.year}" / f"{time.month:02d}" / f"{time.day:02d}" / output_filename
    output_path.parent.mkdir(parents=True, exist_ok=True)
    return output_path


def geotiff_from_dataset(ds: xr.Dataset, var: str, output_root_path: Path, clobber: bool = False) -> Path:
    """
    write geotiff of var from ds to output path
    """
    output_path = create_geotiff_output_path(ds=ds, var=var, output_root_path=output_root_path)
    if clobber and output_path.exists():
        output_path.unlink()
    (
        conus_data[var]
        .astype("float32")
        .rio.to_raster(
            output_path, 
            driver="GTiff", 
            compress="LZW", 
            tiled=True,
            blockxsize=128,
            blockysize=128,
        )
    )
    return output_path


geotiff_path = geotiff_from_dataset(ds=conus_data, var="TMP", output_root_path=Path("./"), clobber=True)

In [ ]:
geotiff_path_str = str(geotiff_path)

In [ ]:
!rio info {geotiff_path_str}

In [ ]:
rds = rioxarray.open_rasterio(
    geotiff_path_str,
    lock=False,  # disable internal caching
    cache=False,  # don't keep data loaded in memory. pull from disk every time
)
rds

In [ ]:
rds.hvplot(cmap='viridis', width=800, height=600, x="x", y="y", t`itle=f'Temperature (K), {np.datetime64(conus_data.time.item(), "ns")}')